# **Business Questions and Hypotheses:**

**1. Does price influence quantity sold? (Main Hypothesis Test)**

- H_1 (Null):There is no statistically significant relationship between product price and quantity sold.
- H_1 (Alternative):There is a statistically significant negative relationship between product price and quantity sold.

**2. Do repeat customers spend more?**

**Approach (Exploratory Data Analysis):** Customer purchasing behavior will be analyzed by segmenting customers into repeat and one-time buyers based on transaction frequency. Total spending will be compared across these groups to identify differences in purchasing patterns.

**3. How do sales trends evolve over time?**

**Approach (Time Series Analysis):** Revenue will be aggregated over time (e.g., monthly) to analyze trends, patterns, and potential seasonality in sales performance.

**4. Crypto Dataset (API): How does cryptocurrency volatility differ across assets?**

**Approach (Exploratory Data Analysis):** Price movements across different cryptocurrencies will be examined to assess and compare volatility levels, providing insight into relative risk and stability among assets.


# **Loading Data**

In [1]:
!pip install kagglehub

In [2]:
import kagglehub

path = kagglehub.dataset_download("gabrielramos87/an-online-shop-business")

print(path)

Using Colab cache for faster access to the 'an-online-shop-business' dataset.
/kaggle/input/an-online-shop-business


In [3]:
import os

os.listdir(path)

['Sales Transaction v.4a.csv']

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from datetime import datetime

df = pd.read_csv(f"{path}/Sales Transaction v.4a.csv")
df.head()

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/9/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490.0,United Kingdom
1,581475,12/9/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069.0,United Kingdom
2,581475,12/9/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069.0,United Kingdom
3,581475,12/9/2019,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069.0,United Kingdom
4,581475,12/9/2019,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069.0,United Kingdom


**#Cleaning and transformation**

In [6]:
#df['Country'].unique()
# removing white space
df['Country'] = df['Country'].str.strip()

In [7]:
# standardise names
df['Country'] = df['Country'].replace({'EIRE': 'Ireland','RSA': 'South Africa'
})

In [8]:
# invalid entries
df = df[~df['Country'].isin(['Unspecified', 'European Community'])]

In [9]:
df['Country'].unique()

array(['United Kingdom', 'Norway', 'Belgium', 'Germany', 'France',
       'Austria', 'Netherlands', 'Ireland', 'USA', 'Channel Islands',
       'Iceland', 'Portugal', 'Spain', 'Finland', 'Italy', 'Greece',
       'Japan', 'Sweden', 'Denmark', 'Cyprus', 'Malta', 'Switzerland',
       'Australia', 'Czech Republic', 'Poland', 'Hong Kong', 'Singapore',
       'South Africa', 'Israel', 'United Arab Emirates', 'Canada',
       'Bahrain', 'Brazil', 'Saudi Arabia', 'Lebanon', 'Lithuania'],
      dtype=object)

In [11]:
#Data dtype
df['Date'] = pd.to_datetime(df['Date'])
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

In [13]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

## Filtering

In [17]:
# filtering out negative values - related to returns
df = df[df['Quantity'] > 0]

In [18]:
(df['Quantity'] < 0).sum()

np.int64(0)

In [21]:
df[df['Price'] < 0]

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country,Year,Month


In [22]:
df['Price'].describe()

,Price
count,527261.000000
mean,12.629106
std,7.936336
min,5.130000
25%,10.990000
50%,11.940000
75%,14.090000
max,660.620000


## Missing values

In [24]:
# unique identifier missing 0.01% of data, we can drop

df = df.dropna(subset=['CustomerNo'])

In [25]:
#making sure it is an integer
df['CustomerNo'] = df['CustomerNo'].astype(int)

In [28]:
#df.isnull().sum()


In [30]:
#df['ProductName'].unique()
#remove white space
df['ProductName'] = df['ProductName'].str.strip()

In [36]:
# adding revenue column for easy analysis.
df['Revenue'] = df['Price'] * df['Quantity']
df

,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country,Year,Month,Revenue
0,581482,2019-12-09,22485,Set Of 2 Wooden Market Crates,21.47,12,17490,United Kingdom,2019,12,257.64
1,581475,2019-12-09,22596,Christmas Star Wish List Chalkboard,10.65,36,13069,United Kingdom,2019,12,383.40
2,581475,2019-12-09,23235,Storage Tin Vintage Leaf,11.53,12,13069,United Kingdom,2019,12,138.36
3,581475,2019-12-09,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069,United Kingdom,2019,12,127.80
4,581475,2019-12-09,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069,United Kingdom,2019,12,71.64
...,...,...,...,...,...,...,...,...,...,...,...
536320,536585,2018-12-01,37449,Ceramic Cake Stand + Hanging Cakes,20.45,2,17460,United Kingdom,2018,12,40.90
536321,536590,2018-12-01,22776,Sweetheart 3 Tier Cake Stand,20.45,1,13065,United Kingdom,2018,12,20.45
536322,536590,2018-12-01,22622,Box Of Vintage Alphabet Blocks,20.45,2,13065,United Kingdom,2018,12,40.90
536323,536591,2018-12-01,37449,Ceramic Cake Stand + Hanging Cakes,20.45,1,14606,United Kingdom,2018,12,20.45


In [33]:
#df.info()

In [41]:
df.duplicated().sum()

np.int64(0)

In [40]:
df = df.drop_duplicates()

# Save cleaned and transformed dataset

In [43]:
df.to_csv('clean_ecommerce_data.csv.gz', index=False, compression='gzip')

#